## Imports and Setup:

In [ ]:
# Requirements: torch, torch_geometric, scikit-learn, numpy
# pip install torch torch_geometric scikit-learn numpy

import os
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import TUDataset, GNNBenchmarkDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.explain import Explainer, PGExplainer, GNNExplainer
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === Configuration ===
BASE_DIR = '/content/drive/MyDrive/GNN_MEA/'
VICTIM_DIR = os.path.join(BASE_DIR, 'victim_models')
EXPLAINER_DIR = os.path.join(BASE_DIR, 'explainers')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

ARCHITECTURES = ['GCN', 'GAT', 'GraphSAGE']

# Create all directories
for arch in ARCHITECTURES:
    os.makedirs(os.path.join(VICTIM_DIR, arch), exist_ok=True)
    os.makedirs(os.path.join(EXPLAINER_DIR, 'PGExplainer', arch), exist_ok=True)
    os.makedirs(os.path.join(EXPLAINER_DIR, 'GNNExplainer_cache', arch), exist_ok=True)

os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Setting up Datasets (importing + train/shadow/test split)

In [ ]:
# Wrapper for non-TUDataset sources (MNIST)
class DatasetWrapper:
    def __init__(self, data_list, n_classes, name):
        self._data = data_list
        self._num_classes = n_classes
        self.name = name
    def __len__(self): return len(self._data)
    def __getitem__(self, idx): return self._data[idx]
    @property
    def num_classes(self): return self._num_classes

In [ ]:
# === TUDataset datasets ===
# Value indicates whether to use use_node_attr
TU_DATASETS = {
    'AIDS': False,
    'MUTAG': False,
    'PTC_FM': False,
    'NCI1': False,
    'Tox21_AhR_training': False,
    'Letter-low': True,
    'Synthie': True,
}

datasets = {}
for name, use_attr in TU_DATASETS.items():
    ds = TUDataset(root=f'data/{name}', name=name, use_node_attr=use_attr)
    datasets[name] = ds
    assert ds[0].x is not None, f"{name}: features still None!"
    print(f"{name}: {len(ds)} graphs, {ds.num_classes} classes, "
          f"feature dim = {ds[0].x.shape[1]}")

# === GNNBenchmarkDataset (MNIST) ===
all_data = []
for split in ['train', 'val', 'test']:
    ds_split = GNNBenchmarkDataset(root='data/MNIST', name='MNIST', split=split)
    all_data.extend([ds_split[i] for i in range(len(ds_split))])

n_classes = len(set(d.y.item() for d in all_data))
datasets['MNIST'] = DatasetWrapper(all_data, n_classes, 'MNIST')
print(f"MNIST: {len(all_data)} graphs, {n_classes} classes, "
      f"feature dim = {all_data[0].x.shape[1]}")

# === Master list ===
DATASET_NAMES = list(datasets.keys())
print(f"\nAll datasets: {DATASET_NAMES}")

In [ ]:
# Stratified 60% train / 20% shadow / 20% test split (deterministic via seed)
def split_dataset(dataset, seed=42):
    labels = [data.y.item() for data in dataset]
    train_idx, remaining_idx = train_test_split(
        range(len(dataset)), test_size=0.4,
        stratify=labels, random_state=seed
    )
    remaining_labels = [labels[i] for i in remaining_idx]
    shadow_idx, test_idx = train_test_split(
        remaining_idx, test_size=0.5,
        stratify=remaining_labels, random_state=seed
    )
    return train_idx, shadow_idx, test_idx


def get_feature_dim(dataset):
    """Reads actual feature dim from data (handles manually added features)."""
    return dataset[0].x.shape[1]

## GCN, GAT, and GraphSAGE Victim Model Architectures:

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.classifier(x)
        return x


class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim // 8, heads=8)
        self.conv2 = GATConv(hidden_dim, hidden_dim // 8, heads=8)
        self.conv3 = GATConv(hidden_dim, hidden_dim // 8, heads=8)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.classifier(x)
        return x


class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.classifier(x)
        return x


MODEL_CLASSES = {'GCN': GCN, 'GAT': GAT, 'GraphSAGE': GraphSAGE}

## Training or Loading Victim Models:

In [ ]:
def check_predictions(model, loader, device):
    """Checks victim predicts multiple classes (not collapsed to one)."""
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            preds.extend(out.argmax(dim=1).cpu().tolist())
    unique, counts = np.unique(preds, return_counts=True)
    print(f"  Prediction distribution: {dict(zip(unique, counts))}")
    if len(unique) == 1:
        print("  WARNING: Model predicts single class!")
    return len(unique) > 1


def load_victim(name, arch, dataset, device='cuda'):
    """Load saved victim model checkpoint."""
    save_path = os.path.join(VICTIM_DIR, arch, f'{name}_victim.pt')
    checkpoint = torch.load(save_path, map_location=device, weights_only=False)
    ModelClass = MODEL_CLASSES[arch]
    model = ModelClass(
        checkpoint['in_dim'],
        checkpoint['config']['hidden_dim'],
        checkpoint['num_classes']
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

In [ ]:
LARGE_DATASETS = {'MNIST'}

def train_victim(dataset, train_idx, test_idx, arch='GCN', device='cuda'):
    train_set = [dataset[i] for i in train_idx]
    test_set = [dataset[i] for i in test_idx]

    for data in train_set + test_set:
        data.y = data.y.long()

    in_dim = get_feature_dim(dataset)
    num_classes = dataset.num_classes
    ModelClass = MODEL_CLASSES[arch]

    # Reduced grid for large datasets
    ds_name = getattr(dataset, 'name', '')
    if ds_name in LARGE_DATASETS or len(dataset) > 10000:
        epoch_choices = [200, 500]
        hidden_choices = [128]
    else:
        epoch_choices = [300, 500, 700, 1000]
        hidden_choices = [64, 128]

    best_acc = 0
    best_model = None
    best_config = {}

    for hidden_dim in hidden_choices:
        for max_epochs in epoch_choices:
            model = ModelClass(in_dim, hidden_dim, num_classes).to(device)
            optimizer = torch.optim.Adam(
                model.parameters(), lr=0.001, weight_decay=5e-4)
            loss_fn = nn.CrossEntropyLoss()
            train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
            test_loader = DataLoader(test_set, batch_size=32)

            model.train()
            for epoch in range(max_epochs):
                for batch in train_loader:
                    batch = batch.to(device)
                    pred = model(batch.x, batch.edge_index, batch.batch)
                    loss = loss_fn(pred, batch.y)
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()

            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(device)
                    pred = model(batch.x, batch.edge_index, batch.batch)
                    correct += (pred.argmax(1) == batch.y).sum().item()
                    total += batch.y.size(0)

            acc = correct / total
            print(f"    hidden={hidden_dim}, epochs={max_epochs}: "
                  f"acc={acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_model = model
                best_config = {'hidden_dim': hidden_dim, 'epochs': max_epochs}

    test_loader = DataLoader(test_set, batch_size=32)
    is_valid = check_predictions(best_model, test_loader, device)
    print(f"  Best: {best_config}, acc={best_acc:.4f}, valid={is_valid}")
    return best_model, best_config, best_acc

In [ ]:
# Train all architectures x all datasets, skip if already saved
victim_models = {}
victim_configs = {}
dataset_splits = {}

for name in DATASET_NAMES:
    ds = datasets[name]
    train_idx, shadow_idx, test_idx = split_dataset(ds)
    dataset_splits[name] = {
        'train': train_idx, 'shadow': shadow_idx, 'test': test_idx}

    for arch in ARCHITECTURES:
        key = f"{name}_{arch}"
        print(f"\n{'='*50}")
        print(f"{arch} on {name}")
        print(f"{'='*50}")

        save_path = os.path.join(VICTIM_DIR, arch, f'{name}_victim.pt')

        if os.path.exists(save_path):
            print(f"  Already trained, loading...")
            model, checkpoint = load_victim(name, arch, ds, device)
            victim_models[key] = model
            victim_configs[key] = checkpoint['config']
            print(f"  Config: {checkpoint['config']}, "
                  f"acc={checkpoint['accuracy']:.4f}")
            continue

        print(f"  Split: {len(train_idx)} train, "
              f"{len(shadow_idx)} shadow, {len(test_idx)} test")

        model, config, acc = train_victim(
            ds, train_idx, test_idx, arch=arch, device=device)
        victim_models[key] = model
        victim_configs[key] = config

        # Save model + config + splits
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': config,
            'accuracy': acc,
            'in_dim': get_feature_dim(ds),
            'num_classes': ds.num_classes,
            'arch': arch,
            'train_idx': train_idx,
            'shadow_idx': shadow_idx,
            'test_idx': test_idx,
        }, save_path)
        print(f"  Saved to {save_path}")

print("\n\nSummary:")
print("-" * 60)
for key in sorted(victim_configs.keys()):
    print(f"  {key}: {victim_configs[key]}")

## Training or Loading Explainers:

In [ ]:
def train_pg_explainer(model, dataset, train_idx, name, arch, device='cuda'):
    """Train PGExplainer (edge masks) for 100 epochs."""
    explainer = Explainer(
        model=model,
        algorithm=PGExplainer(epochs=100, lr=0.003),
        explanation_type='phenomenon',
        edge_mask_type='object',
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )
    explainer.algorithm = explainer.algorithm.to(device)

    for epoch in range(100):
        total_loss = 0
        for idx in train_idx:
            data = dataset[idx].to(device)
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
            target = model(data.x, data.edge_index, batch).argmax(dim=1)
            loss = explainer.algorithm.train(
                epoch, model, data.x, data.edge_index,
                target=target, batch=batch,
            )
            total_loss += loss
        if epoch % 25 == 0:
            print(f"    Epoch {epoch}: loss={total_loss:.4f}")

    save_path = os.path.join(EXPLAINER_DIR, 'PGExplainer', arch,
                              f'{name}_explainer.pt')
    torch.save(explainer.algorithm.state_dict(), save_path)
    print(f"  Saved to {save_path}")
    return explainer


def create_gnn_explainer(model):
    """GNNExplainer uses node masks; deepcopy prevents corrupting victim."""
    model_copy = copy.deepcopy(model)
    for param in model_copy.parameters():
        param.requires_grad_(True)
    return Explainer(
        model=model_copy,
        algorithm=GNNExplainer(epochs=100, lr=0.01),
        explanation_type='phenomenon',
        node_mask_type='object',
        edge_mask_type=None,
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

In [ ]:
def get_explanation(explainer, model, data, device='cuda'):
    """Get explanation: handles edge masks (PGExp) and node masks (GNNExp)."""
    data = data.to(device)
    batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    target = model(data.x, data.edge_index, batch).argmax(dim=1)

    explanation = explainer(data.x, data.edge_index,
                            target=target, batch=batch)

    # Node masks (GNNExplainer)
    if hasattr(explanation, 'node_mask') and explanation.node_mask is not None:
        node_mask = explanation.node_mask
        node_importance = node_mask.mean(dim=1) if node_mask.dim() > 1 else node_mask
        threshold = node_importance.median()
        important_nodes = (node_importance > threshold).nonzero(as_tuple=True)[0]
        src, dst = data.edge_index
        edge_mask_bool = (torch.isin(src, important_nodes) &
                          torch.isin(dst, important_nodes))
        important_edges = edge_mask_bool.nonzero(as_tuple=True)[0]
        return important_nodes.cpu(), important_edges.cpu(), node_importance.cpu()

    # Edge masks (PGExplainer)
    elif hasattr(explanation, 'edge_mask') and explanation.edge_mask is not None:
        edge_mask = explanation.edge_mask
        threshold = edge_mask.median()
        important_edges = (edge_mask > threshold).nonzero(as_tuple=True)[0]
        important_nodes = torch.unique(
            data.edge_index[:, important_edges].flatten())
        return important_nodes.cpu(), important_edges.cpu(), edge_mask.cpu()

    else:
        raise ValueError("No mask found in explanation")


def verify_explanations(explainer, model, dataset, sample_idx, device='cuda'):
    """Verify explanations aren't trivial (checks node count thresholds)."""
    n_trivial = 0
    n_tested = min(20, len(sample_idx))

    for idx in sample_idx[:n_tested]:
        imp_nodes, imp_edges, mask = get_explanation(
            explainer, model, dataset[idx], device)

        total_nodes = dataset[idx].num_nodes
        pct_important = len(imp_nodes) / total_nodes * 100

        # Trivial if <5% or >95% nodes marked important
        if pct_important < 5 or pct_important > 95:
            n_trivial += 1

    pct_trivial = n_trivial / n_tested * 100
    print(f"  Verified: {pct_trivial:.0f}% trivial explanations "
          f"({n_trivial}/{n_tested})")
    return pct_trivial < 50

In [ ]:
# Train PGExplainer for all architecture x dataset pairs
explainers = {}

for name in DATASET_NAMES:
    ds = datasets[name]
    splits = dataset_splits[name]

    for arch in ARCHITECTURES:
        key = f"{name}_{arch}"
        if key not in victim_models:
            continue

        model = victim_models[key]

        # --- PGExplainer (trained, saves to disk) ---
        pg_key = f"{key}_PG"
        pg_path = os.path.join(EXPLAINER_DIR, 'PGExplainer', arch,
                                f'{name}_explainer.pt')

        print(f"\n{'='*50}")
        print(f"PGExplainer: {arch} on {name}")
        print(f"{'='*50}")

        if os.path.exists(pg_path):
            print(f"  Already trained, loading...")
            pg_explainer = Explainer(
                model=model,
                algorithm=PGExplainer(epochs=100, lr=0.003),
                explanation_type='phenomenon',
                edge_mask_type='object',
                model_config=dict(
                    mode='multiclass_classification',
                    task_level='graph',
                    return_type='raw',
                ),
            )
            pg_explainer.algorithm = pg_explainer.algorithm.to(device)
            pg_explainer.algorithm.load_state_dict(
                torch.load(pg_path, weights_only=False))
            pg_explainer.algorithm._curr_epoch = 99
        else:
            pg_explainer = train_pg_explainer(
                model, ds, splits['train'], name, arch, device)

        # Verify explanations aren't trivial
        is_valid = verify_explanations(
            pg_explainer, model, ds, splits['shadow'], device)
        print(f"  Valid: {is_valid}")
        explainers[pg_key] = pg_explainer